# ANEEL RAG — Etapa 3 · Chunking Hierárquico 2016
### Processa os markdowns extraídos na Etapa 2

> Rode após a conclusão da Etapa 2.
> Se a sessão cair, reexecute a Célula 7 — retoma via checkpoint.

**Entrada:**
- `extracted/{ano}/doc_id.md`
- `extracted/{ano}/doc_id.meta.json`

**Saída:**
- `chunks/{ano}/doc_id.chunks.json`
- `logs/chunking_{ano}.csv`

**Níveis de chunk:**
| Nível | Descrição | Tokens alvo |
|---|---|---|
| L0 | Ementa / título oficial do documento | 20–600 |
| L1 | Artigo jurídico completo ou seção | 20–800 |
| L2 | Parágrafo com overlap de 60 tokens | 20–400 |

**Estratégias por tipo de documento:**
| Tipo | Splitting L1 | Coerência mínima |
|---|---|---|
| Digital ok_gate | Por artigos (`Art. X`) | 0.75 |
| Digital ok_gate sem artigos | Por seções Markdown (`##`) | 0.75 |
| OCR ok_gate | Por artigos ou seções | 0.50 |
| Degradado (ok_degradado) | Por parágrafos | 0.40 |
| Ofício/Carta sem artigos | Por parágrafos | 0.40 |

In [ ]:
# Célula 0 — Geração automática de vocabulário a partir do corpus
# Execute UMA VEZ antes das outras células.
# Lê todos os .md extraídos e constrói o vocabulário do domínio automaticamente.

from google.colab import drive
import re, json, unicodedata
from pathlib import Path
from collections import Counter

# Monta Drive se necessário
try:
    Path('/content/drive/MyDrive').exists()
except:
    drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/aneel_rag')
ANO  = '2016'

print(f'Gerando vocabulário automático a partir dos documentos extraídos...')

# Coleta todas as palavras do corpus
all_words = Counter()
mds = list((BASE / 'extracted' / ANO).glob('*.md'))
print(f'Lendo {len(mds):,} documentos...')

for i, md in enumerate(mds):
    try:
        texto = md.read_text(encoding='utf-8')
        # Remove markdown, links, números, símbolos
        texto = re.sub(r'http\S+', '', texto)
        texto = re.sub(r'<!--.*?-->', '', texto, flags=re.DOTALL)
        texto = re.sub(r'[\|\-\_\*\#\[\]\(\)\{\}]', ' ', texto)
        words = re.findall(r'[a-zA-ZáéíóúâêôàãõçÁÉÍÓÚÂÊÔÀÃÕÇ]{4,}', texto.lower())
        all_words.update(words)
    except:
        pass
    if (i+1) % 1000 == 0:
        print(f'  {i+1:,}/{len(mds):,} processados...')

print(f'Total de palavras únicas: {len(all_words):,}')

# Filtra: mantém palavras que aparecem em pelo menos 3 documentos
# e remove stopwords universais
STOPWORDS = set('''
    para como mais mais pelo pela pelos pelas esse essa esses essas este
    esta estes estas isso isto aqui aquela aquele aquelas aqueles
    muito muita muitos muitas sendo sendo tendo tendo haver
    fazer fazer dever poder querer qual quando onde
    http www null true false none type text page
    versao numero data item
'''.split())

def normalizar(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii').lower()

# Pega as top 3000 palavras mais frequentes, excluindo stopwords
vocab_automatico = set()
for palavra, freq in all_words.most_common(5000):
    if freq < 3:
        break
    palavra_norm = normalizar(palavra)
    if palavra_norm not in STOPWORDS and len(palavra_norm) >= 4:
        vocab_automatico.add(palavra_norm)
        vocab_automatico.add(palavra)  # versão com acento também

# Salva no Drive para reutilizar
vocab_path = BASE / 'logs' / f'vocab_2016.json'
vocab_path.parent.mkdir(exist_ok=True)
with open(vocab_path, 'w') as f:
    json.dump(sorted(vocab_automatico), f, ensure_ascii=False)

print(f'Vocabulário gerado: {len(vocab_automatico):,} termos')
print(f'Salvo em: {vocab_path}')
print(f'\nTop 20 termos mais frequentes do corpus:')
for w, n in all_words.most_common(20):
    if normalizar(w) not in STOPWORDS:
        print(f'  {w}: {n:,}x')


In [ ]:
# Célula 1 — Dependências
!pip install tiktoken --quiet

import tiktoken
import torch
print('Etapa 3: Chunking Hierárquico v2')
print(f'GPU disponível: {torch.cuda.is_available()}')

In [ ]:
# Célula 2 — Drive e estrutura
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, re, json, time

BASE = Path('/content/drive/MyDrive/aneel_rag')
ANO = '2016'

for ano in ["2016"]:
    (BASE / 'chunks' / ano).mkdir(parents=True, exist_ok=True)
(BASE / 'logs').mkdir(parents=True, exist_ok=True)

for ano in ["2016"]:
    n = len(list((BASE / 'extracted' / ano).glob('*.md')))
    print(f'Ano {ano}: {n:,} markdowns disponíveis')

In [ ]:
# Célula 3 — Configurações e parâmetros
import tiktoken, re
_RE_PALAVRAS = re.compile(r'[a-zA-ZáéíóúâêôàãõçÁÉÍÓÚÂÊÔÀÃÕÇ]{4,}')
_RUIDO = frozenset(['http', 'www', 'null', 'true', 'false', 'none'])

TOKENIZER = tiktoken.get_encoding('cl100k_base')

CHUNK_CONFIG = {
    'L0': {'min_tokens': 20,  'max_tokens': 600},
    'L1': {'min_tokens': 20,  'max_tokens': 800},
    'L2': {'min_tokens': 20,  'max_tokens': 450, 'overlap_tokens': 100},
}

COERENCIA_MIN = {
    'ok_gate':      0.75,
    'ok_fallback':  0.75,
    'ok_degradado': 0.40,
    'ocr':          0.50,
    'geo':          0.0,   # memorial descritivo/coordenadas: bypass total
}

GATE = {'min_cobertura': 1.00, 'checkpoint_every': 100}
CHECKPOINT_EVERY = GATE['checkpoint_every']

_vocab = (
    'de do da dos das em no na nos nas para por com um uma o a os as '
    'que se ao aos pela pelo pelos pelas este esta estes estas esse '
    'essa esses essas nao sim art artigo paragrafo inciso lei decreto '
    'resolucao portaria despacho publicado federal nacional diario '
    'oficial agencia energia eletrica aneel considerando resolve '
    'determina estabelece dispoe processo interessado numero data '
    'autorizacao concessao permissao servico sistema ministerio '
    'resolucao normativa despacho portaria autorizativa superintendencia '
    'megawatt quilowatt kilovolt mwh kwh mw kv gw tensao '
    'transmissao distribuicao geracao comercializacao '
    'contrato tarifa consumidor usuario titular agente '
    'instalacao subestacao linha usina hidreletrica '
    'termoeletrica eolica fotovoltaica potencia capacidade '
    'homologar homologacao revisao tarifaria periodica '
    'reajuste vigencia aplicacao subgrupo modalidade '
    'voto relatorio fundamentacao direito dispositivo '
    'oficio pleito cooperativa distribuidora permissionaria '
    'ons ccee mme tust tusd rap proret renic '
    'transmissora geradora comercializadora '
    'poligono coordenadas latitude longitude utm norte sul leste oeste '
    'area hectare perimetro municipio estado bacia rio lago acude '
    'reservatorio barragem vertedor turbina gerador montante jusante '
    'pch uhe ute aproveitamento hidreletrico azimute distancia vertice '
    'confrontando sirgas datum app dup georeferenciamento '
    'destinacao preservacao permanente remanescente faixa margem '
)

_vocab_extra = (
    'conforme anexo calculo submodulo periodo desconto valor tarifario tarifaria tarifarias base parcela revisao consumidores fiscalizacao servicos tarifas repasse receita infracao aquicultura fonte incentivada mercado encargos montantes teto item investimentos qualidade redes anual referente bandeiras recurso apuracao correcao anterior previsao termo crescimento recursos componentes cofins faturas simulacao multa superintendente informacoes assunto distribuidora concessionaria concessionario concessoes permissoes outorgas interposto administrativo lavrado requerimento requerente requerida solicitacao dosimetria penalidade penalidades sancao sancoes infrações fiscalizado fiscaliza notificacao auto encerramento revogou reincidencia abrangencia gravidade conformidade contabilizacao normas procedimentos instrucoes regulamento setor indisponibilidade publico principio administrador aplicado aplicar aplicacao percentual enquadramento procuradoria resultado consideracoes avaliacao peso operacional liquida rol faturamento dezembro novembro outubro setembro agosto janeiro fevereiro marco abril junho julho meses meses anos periodo ciclo eletricidade irrigacao rural agricola cooperativa cooperantes cooperados socios fates fundo assistencia tecnica educacional social estatuto assembleia geral extraordinaria ordinaria aprovado aprovados votacao deliberacao deliberou sobras perdas tecninas nao tecnicas simplificado nota tecnica determinado determinacao fixacao fixado fixar estipulado estipular observancia disposicoes dispositivos amparo legal embasamento fundamento documentos apresentados conformidade requisitos normativos recomendacao favoravel desfavoravel analise exame verificacao constatado constatacao deferimento indeferimento provimento improvimento recurso recorrente recorrida proret srg sfe srd srh sge sgt sfg sic agi dir concessionaria distribuidora transmissora geradora comercializadora energia_eletrica setor eletrico sistema_interligado nacional garantia fisica lastro contrato ccear ccee oficio circular manual samp acompanhamento informacao regulacao economica custo operacional investimento expansao melhoria planejamento obras depreciacao amortizacao ativo passivo patrimonio capital proprio ipca igpm selic variacao acumulada indice correcao monetaria tributos impostos contribuicoes pis cofins icms iss csll irpj subgrupo modalidade tarifaria tensao alta media baixa rural irrigante aquicultura classe consumidores especiais medicao faturamento consumo demanda potencia ativa reativa '
)
_vocab_extra2 = (
    'publica participacao audiencia envio contribuicoes enderecos tema objeto limites equivalente interrupcao unidade consumidora aviso audiencias publicas indicadores continuidade fec dec frequencia duracao subsidios aprimoramento documentacao modelo criterios procedimentos periodo contribuicao intercambio documental comunicado objeto acesso canal definidos publicadas internet menu principal item cargo comissionado tecnico servidora matricula siape exoneracao nomeacao cargos servidores ocupantes efetivos quadro pessoal especifico extincao carreira procurador federal analista administrativo coordenacao capacitacao gestao conhecimento reestruturacao atividade adjunta projeto exonerar nomear ocupantes premissas destinado desempenho diante decido contar publicacao portaria respectiva sgan quadra modulo terreo protocolo brasiliadf correio eletronico mediacao ouvidoria setorial competencia atribuida superintendente texto original comunica aberta obter verbis transcrito abaixo cabe destacar premissas atende disposto prevista pedido proprio servidor juizo autoridade competente base legal diante exposto decido contar publicacao exonerado exonerada nomeado nomeada ficando mesma data '
)
# Constrói vocab com normalização de acentos
import unicodedata as _ud
def _norm(s): return _ud.normalize('NFKD', s).encode('ascii','ignore').decode('ascii')
_vocab_contabil = (
    'conta contabilidade subconta manual eletrico contrapartida transferencia ativo credito debito bens saldo imobilizado circulante subcontas exercicio outorgada passivo provisao grupo capital administracao contas ativos funcao titulo amortizacao juros codigo contabeis social gastos atividades apropriadas creditos subsistema patrimonial destina acumulada financeiros despesas intangivel devera receitas balancete trimestral prestacao socioambiental obrigatoriedade elaboracao relatorios resumo exercicio arquivos instrucoes gerais aplicabilidade conceitos fundamentos estruturacao prefacio introducao objetivos diretrizes premissas contabilizacao estrutura cadastro controle direitos balanco demonstracao resultado patrimonio liquido fluxo caixa notas explicativas auditoria independente consolidado requerida solicitada aprovacao homologada fixada estabelecida determinada vigente anterior posterior subsequente subgrupo tensao modalidade classe consumidor especial bandeira tarifaria verde amarela vermelha escassez componente financeiro neutralidade tfsee cde proinfa ess encargo uso transmissao distribuicao conexao parcela ajuste reposicionamento fator produtividade receita permitida requerida verificada custo marginal expansao desenvolvimento suprimento exposicao motivos presidente republica ministerio minas conselho politica energetica cnpe resolucao aprovo despacho presidencia republica '
)
# Carrega vocab automático do Drive
import json as _js
_vocab_auto = ''
_vp = Path('/content/drive/MyDrive/aneel_rag/logs/vocab_2016.json')
if _vp.exists():
    _vocab_auto = ' '.join(_js.load(open(_vp)))
    print(f'Vocab automático: {len(_vocab_auto.split()):,} termos')
else:
    print('Vocab automático não encontrado, usando apenas vocab manual')
_raw = set((_vocab + ' ' + _vocab_extra + ' ' + _vocab_extra2 + ' ' + _vocab_contabil + ' ' + _vocab_auto).split())
# Pré-normaliza todo o vocab uma única vez — lookup O(1)
VOCAB_PTBR = _raw | {_norm(t) for t in _raw}
# Compila regex de palavras uma única vez
_RE_PALAVRAS = re.compile(r'[a-zA-ZáéíóúâêôàãõçÁÉÍÓÚÂÊÔÀÃÕÇ]{4,}')

def contar_tokens(texto: str) -> int:
    return len(TOKENIZER.encode(texto))

def calcular_coerencia(texto: str) -> float:
    if not texto or not texto.strip():
        return 0.0
    # Filtra só palavras reais (min 4 chars, sem números)
    tok = [t for t in _RE_PALAVRAS.findall(texto.lower()) if t not in _RUIDO]
    if not tok:
        return 0.0
    return sum(1 for t in tok if t in VOCAB_PTBR) / len(tok)

print('Config v13 carregada')
print(f'  COERENCIA_MIN geo = {COERENCIA_MIN["geo"]}')
print(f'  Vocab: {len(VOCAB_PTBR)} termos')


In [ ]:
# Célula 4 — Limpeza e detecção de tipo
RE_ARTIGO = re.compile(r'(?=^Art\.?\s*\d+|^ARTIGO\s+\d+)', re.IGNORECASE | re.MULTILINE)
RE_TABELA = re.compile(r'^\|', re.MULTILINE)
RE_IMAGE  = re.compile(r'<!--\s*image\s*-->', re.IGNORECASE)
RE_TITULO = re.compile(
    r'^(?:##?\s*)?(?:RESOLU[ÇC][ÃA]O|DESPACHO|PORTARIA|INSTRU[ÇC][ÃA]O|'
    r'DELIBERA[ÇC][ÃA]O|HOMOLOGAT[OÓ]RIA|AUTORIZATIVA|NORMATIVA|LEI|DECRETO)'
    r'[^\n]{0,200}',
    re.IGNORECASE | re.MULTILINE
)
RE_GEO = re.compile(
    r'(?:area|pol[ií]gono|coordenadas|shapefile|memorial|barragem|'
    r'reservat[oó]rio|vertedor|montante|jusante|hectare|georref|_dup)',
    re.IGNORECASE
)

def limpar_texto(texto: str) -> str:
    texto = RE_IMAGE.sub('', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    texto = re.sub(r' {2,}', ' ', texto)
    return texto.strip()

def detectar_tipo_doc(meta: dict, nome_arquivo: str = '') -> str:
    status    = meta.get('extrator', 'docling')
    ocr_usado = meta.get('ocr_usado', False)
    if 'degradado' in status:
        return 'degradado'
    if ocr_usado:
        return 'ocr'
    nome = (meta.get('arquivo', '') + ' ' + nome_arquivo).lower()
    if RE_GEO.search(nome):
        return 'geo'
    # Exposição de motivos — documento muito curto, tratado como degradado
    if re.search(r'exm|exposicao', nome):
        return 'degradado'
    return 'normal'

def coerencia_minima(tipo_doc: str) -> float:
    return COERENCIA_MIN.get(tipo_doc, COERENCIA_MIN['ok_gate'])

def tem_tabela(texto: str) -> bool:
    return bool(RE_TABELA.search(texto))

def extrair_titulo_oficial(texto: str) -> str:
    m = RE_TITULO.search(texto)
    if m:
        trecho = texto[m.start():m.start()+400]
        return '\n'.join(trecho.split('\n')[:3]).strip()
    return ''

print('Funções de limpeza e detecção v6 carregadas')


In [ ]:
# Célula 5 — Chunking L0/L1/L2
def split_respeitando_tabelas(texto: str, max_tokens: int, overlap: int) -> list:
    linhas = texto.split('\n')
    blocos, atual = [], []
    em_tabela = False
    tokens_atual = 0
    for linha in linhas:
        e_tabela = linha.strip().startswith('|')
        tokens_linha = contar_tokens(linha)
        if e_tabela:
            em_tabela = True
        if tokens_atual + tokens_linha > max_tokens and not em_tabela and atual:
            blocos.append('\n'.join(atual))
            overlap_linhas, overlap_tok = [], 0
            for l in reversed(atual):
                t = contar_tokens(l)
                if overlap_tok + t > overlap:
                    break
                overlap_linhas.insert(0, l)
                overlap_tok += t
            atual = overlap_linhas
            tokens_atual = overlap_tok
        atual.append(linha)
        tokens_atual += tokens_linha
        if em_tabela and not e_tabela and linha.strip() != '':
            em_tabela = False
    if atual:
        blocos.append('\n'.join(atual))
    return [b.strip() for b in blocos if b.strip()]

def extrair_l0(texto: str, meta: dict) -> dict:
    titulo = extrair_titulo_oficial(texto)
    if not titulo:
        linhas, cabecalho = texto.strip().split('\n'), []
        for linha in linhas:
            if RE_ARTIGO.match(linha): break
            cabecalho.append(linha)
            if len(cabecalho) >= 15: break
        titulo = '\n'.join(cabecalho).strip()
    if not titulo:
        titulo = '\n'.join(texto.strip().split('\n')[:5]).strip()
    tokens = contar_tokens(titulo)
    if tokens > CHUNK_CONFIG['L0']['max_tokens']:
        enc = TOKENIZER.encode(titulo)
        titulo = TOKENIZER.decode(enc[:CHUNK_CONFIG['L0']['max_tokens']])
        tokens = CHUNK_CONFIG['L0']['max_tokens']
    return {
        'chunk_id': f"{meta['doc_id']}__L0", 'doc_id': meta['doc_id'],
        'ano': meta['ano'], 'nivel': 'L0', 'indice': 0,
        'conteudo': titulo, 'tokens': tokens,
        'coerencia': round(calcular_coerencia(titulo), 3),
        'artigo_ref': None, 'tem_tabela': False,
    }

def extrair_l1(texto: str, meta: dict, tipo_doc: str) -> list:
    partes_art = [p.strip() for p in RE_ARTIGO.split(texto) if p.strip()]
    if len(partes_art) >= 2:
        partes, estrategia = partes_art, 'artigos'
    else:
        secoes = [s.strip() for s in re.split(r'(?=^#{1,4}\s)', texto, flags=re.MULTILINE) if s.strip()]
        if len(secoes) >= 2:
            partes, estrategia = secoes, 'secoes'
        else:
            partes, estrategia = [p.strip() for p in texto.split('\n\n') if p.strip()], 'paragrafos'
    chunks = []
    for i, parte in enumerate(partes):
        tokens = contar_tokens(parte)
        if tokens < CHUNK_CONFIG['L1']['min_tokens']:
            continue
        if tokens > CHUNK_CONFIG['L1']['max_tokens']:
            enc = TOKENIZER.encode(parte)
            parte = TOKENIZER.decode(enc[:CHUNK_CONFIG['L1']['max_tokens']])
            tokens = CHUNK_CONFIG['L1']['max_tokens']
        m = re.match(r'(Art\.?\s*\d+[°º]?)', parte, re.IGNORECASE)
        chunks.append({
            'chunk_id': f"{meta['doc_id']}__L1_{i:04d}", 'doc_id': meta['doc_id'],
            'ano': meta['ano'], 'nivel': 'L1', 'indice': i,
            'conteudo': parte, 'tokens': tokens,
            'coerencia': round(calcular_coerencia(parte), 3),
            'artigo_ref': m.group(1) if m else None,
            'tem_tabela': tem_tabela(parte), 'estrategia': estrategia,
        })
    return chunks

def extrair_l2(chunks_l1: list, meta: dict) -> list:
    overlap, max_tok, min_tok = CHUNK_CONFIG['L2']['overlap_tokens'], CHUNK_CONFIG['L2']['max_tokens'], CHUNK_CONFIG['L2']['min_tokens']
    chunks_l2 = []
    for chunk_l1 in chunks_l1:
        if chunk_l1['tokens'] <= max_tok:
            c = chunk_l1.copy()
            c['chunk_id'] = chunk_l1['chunk_id'].replace('__L1_', '__L2_') + '_0000'
            c['nivel'] = 'L2'
            c['l1_ref'] = chunk_l1['chunk_id']
            chunks_l2.append(c)
            continue
        blocos = split_respeitando_tabelas(chunk_l1['conteudo'], max_tok, overlap)
        for sub_i, bloco in enumerate(blocos):
            tokens = contar_tokens(bloco)
            if tokens < min_tok:
                continue
            chunks_l2.append({
                'chunk_id': f"{chunk_l1['chunk_id'].replace('__L1_','__L2_')}_{sub_i:04d}",
                'doc_id': meta['doc_id'], 'ano': meta['ano'], 'nivel': 'L2', 'indice': sub_i,
                'conteudo': bloco, 'tokens': tokens,
                'coerencia': round(calcular_coerencia(bloco), 3),
                'artigo_ref': chunk_l1.get('artigo_ref'),
                'tem_tabela': tem_tabela(bloco), 'l1_ref': chunk_l1['chunk_id'],
            })
    return chunks_l2

def gate_chunk(chunk: dict, tipo_doc: str) -> bool:
    """
    Decide se o chunk deve ser mantido ou descartado.
    Aplica limites de coerência diferentes baseados no tipo do documento.
    """
    cfg = CHUNK_CONFIG.get(chunk['nivel'], CHUNK_CONFIG['L2'])

    # 1. Filtro de tamanho mínimo (Se for muito pequeno, descarta)
    if chunk['tokens'] < cfg['min_tokens']:
        return False

    # 2. Bypass para Tabelas (Sempre mantém tabelas, pois o score de texto nelas é baixo)
    if chunk.get('tem_tabela'):
        return True

    # --- LÓGICA DE SENSIBILIDADE DINÂMICA ---
    # Define o limite de coerência conforme a natureza do arquivo
    if tipo_doc == 'geo':
        limite_sobrevivencia = 0.08  # Muito tolerante para coordenadas/polígonos
    elif tipo_doc == 'degradado':
        limite_sobrevivencia = 0.12  # Tolerante para OCR antigo (2016)
    else:
        limite_sobrevivencia = 0.22  # Rigor padrão para textos narrativos

    # 3. Validação final pelo score
    if chunk['coerencia'] < limite_sobrevivencia:
        return False

    return True

print('Funções de chunking v6 carregadas')
print('  gate_chunk: bypass geo ativo quando min_coerencia==0.0')


In [ ]:
# Célula 6 — Função principal de processamento (Ajustada para Filtro Dinâmico)
import json as _json, time

def processar_documento(md_path, meta_path):
    log = {
        'doc_id': md_path.stem, 'status': 'erro',
        'tipo_doc': '', 'estrategia_l1': '',
        'n_l0': 0, 'n_l1': 0, 'n_l2': 0,
        'n_gate_ok': 0, 'n_gate_fail': 0,
        'coerencia_media': 0.0, 'tokens_total': 0,
        'tempo_s': 0.0, 'erro': ''
    }
    t0 = time.time()
    try:
        texto = md_path.read_text(encoding='utf-8')
        meta = _json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {
            'doc_id': md_path.stem, 'ano': md_path.parent.name,
            'extrator': 'docling', 'ocr_usado': False
        }

        if not texto.strip():
            log['erro'] = 'markdown vazio'
            return log, None

        texto = limpar_texto(texto)
        tipo_doc = detectar_tipo_doc(meta, md_path.stem)

        # O Gate agora decide o limite internamente usando o tipo_doc
        chunk_l0 = extrair_l0(texto, meta)
        chunks_l1 = extrair_l1(texto, meta, tipo_doc)

        if not chunks_l1:
            chunks_l1 = [{
                'chunk_id': f"{meta['doc_id']}__L1_0000", 'doc_id': meta['doc_id'],
                'ano': meta['ano'], 'nivel': 'L1', 'indice': 0,
                'conteudo': texto, 'tokens': contar_tokens(texto),
                'coerencia': round(calcular_coerencia(texto), 3),
                'artigo_ref': None, 'tem_tabela': tem_tabela(texto),
                'estrategia': 'documento_inteiro',
            }]

        chunks_l2 = extrair_l2(chunks_l1, meta)
        todos = [chunk_l0] + chunks_l1 + chunks_l2

        # --- MUDANÇA AQUI: Passando tipo_doc para o gate_chunk ---
        aprovados  = [c for c in todos if gate_chunk(c, tipo_doc)]
        reprovados = [c for c in todos if not gate_chunk(c, tipo_doc)]
        # -------------------------------------------------------

        coer_media = sum(c['coerencia'] for c in aprovados) / max(len(aprovados), 1)
        estrategia = chunks_l1[0].get('estrategia', '?') if chunks_l1 else '?'

        saida = {
            'doc_id': meta['doc_id'], 'ano': meta['ano'],
            'arquivo': meta.get('arquivo', md_path.name),
            'extrator': meta.get('extrator', 'docling'),
            'tipo_doc': tipo_doc, 'estrategia_l1': estrategia,
            'score_etapa2': meta.get('score_final', None),
            'ocr_usado': meta.get('ocr_usado', False),
            'n_chunks': {'L0': 1, 'L1': len(chunks_l1), 'L2': len(chunks_l2)},
            'gate': {
                'aprovados': len(aprovados),
                'reprovados': len(reprovados),
                'tipo_aplicado': tipo_doc # Para sabermos qual régua foi usada
            },
            'coerencia_media': round(coer_media, 3),
            'chunks': aprovados, 'reprovados': reprovados,
        }

        log.update({
            'status': 'ok', 'tipo_doc': tipo_doc, 'estrategia_l1': estrategia,
            'n_l0': 1, 'n_l1': len(chunks_l1), 'n_l2': len(chunks_l2),
            'n_gate_ok': len(aprovados), 'n_gate_fail': len(reprovados),
            'coerencia_media': round(coer_media, 3),
            'tokens_total': sum(c['tokens'] for c in aprovados),
            'tempo_s': round(time.time() - t0, 2),
        })
        return log, saida

    except Exception as e:
        log['erro'] = str(e)[:200]
        log['tempo_s'] = round(time.time() - t0, 2)
        return log, None

print('processar_documento v6.1 pronta (com Gate Dinâmico)')

In [ ]:
# Célula 7 — Validação com 5 documentos por ano
# Execute sempre antes da produção!

ANO_TESTE = ANO  # definido na Célula 2
dir_extracted = BASE / 'extracted' / ANO_TESTE
mds = sorted(dir_extracted.glob('*.md'))
if not mds:
    print(f'Ano {ANO_TESTE}: nenhum .md encontrado')
else:
    mds_validos = [p for p in mds if p.stat().st_size > 100]
    peq = sorted(mds_validos, key=lambda p: p.stat().st_size)[:2]
    grd = sorted(mds_validos, key=lambda p: p.stat().st_size, reverse=True)[:2]
    med = [mds_validos[len(mds_validos)//2]]
    amostra = list(dict.fromkeys(peq + med + grd))[:5]

    print('='*65)
    print(f'VALIDAÇÃO {ANO_TESTE} — {len(amostra)} documentos')
    print('='*65)
    resultados = []

    for i, md_path in enumerate(amostra):
        meta_path = md_path.with_suffix('.meta.json')
        kb = md_path.stat().st_size // 1024
        print(f'\n[{i+1}/{len(amostra)}] {md_path.name[:55]:<55} ({kb} KB)')
        log, saida = processar_documento(md_path, meta_path)
        if saida:
            print(f'  Tipo     : {log["tipo_doc"]:<12} | Estratégia L1: {log["estrategia_l1"]}')
            print(f'  Chunks   : L0=1  L1={log["n_l1"]}  L2={log["n_l2"]}')
            print(f'  Gate     : {log["n_gate_ok"]} aprovados / {log["n_gate_fail"]} reprovados')
            print(f'  Coerência média : {log["coerencia_media"]:.3f}')
            print(f'  Tokens total    : {log["tokens_total"]:,}')
            print(f'  Tempo    : {log["tempo_s"]}s')
            ex = next((c for c in saida['chunks'] if c['nivel']=='L1'), None)
            if ex:
                tabela_flag = '[COM TABELA]' if ex.get('tem_tabela') else ''
                print(f'  Ex L1    : {tabela_flag} {ex["conteudo"][:100].strip()}...')
            # Preview de chunks por estrategia
            for estrategia in ['artigos', 'secoes', 'paragrafos']:
                ex = next((c for c in saida['chunks']
                          if c['nivel']=='L1' and c.get('estrategia')==estrategia), None)
                if ex:
                    preview = ex['conteudo'][:120].strip().replace('\n',' ')
                    print(f'  Ex {estrategia:<12}: [{ex["tokens"]} tok] {preview}...')
        else:
            print(f'  ERRO: {log["erro"]}')
        resultados.append(log)

    df_v = pd.DataFrame(resultados)
    n_ok = (df_v['status'] == 'ok').sum()
    total_docs = len(mds_validos)
    est_horas  = df_v['tempo_s'].mean() * total_docs / 3600
    print(f'\nValidação {ANO_TESTE}: {n_ok}/5 OK | Coerência média: {df_v["coerencia_media"].mean():.3f} | Est: {est_horas:.1f}h')
    status_txt = 'APROVADA' if n_ok >= 4 else 'ATENÇÃO'
    print(f'{status_txt} ({n_ok}/5) — {"Avance para Célula 8." if n_ok >= 4 else "Revise antes da produção."}')

In [ ]:
# Célula 8 — Produção: todos os anos
from tqdm.auto import tqdm

def load_log_existente(log_path: Path) -> set:
    if log_path.exists():
        df = pd.read_csv(log_path)
        # Consideramos apenas o que deu 'ok' para permitir reprocessar erros se necessário
        ok = set(df[df['status'] == 'ok']['doc_id'].tolist())
        print(f'Log existente: {len(ok):,} já processados')
        return ok
    return set()

def run_chunking(ano: str):
    dir_extracted = BASE / 'extracted' / ano
    dir_chunks    = BASE / 'chunks'    / ano
    log_path      = BASE / 'logs'      / f'chunking_{ano}.csv'

    dir_chunks.mkdir(parents=True, exist_ok=True)
    (BASE / 'logs').mkdir(parents=True, exist_ok=True)

    ja_ok = load_log_existente(log_path)
    todos = [p for p in sorted(dir_extracted.glob('*.md')) if p.stem not in ja_ok]
    total_ano = len(list(dir_extracted.glob('*.md')))

    print(f'Ano {ano}: total={total_ano:,} | ja_ok={len(ja_ok):,} | pendentes={len(todos):,}')
    if not todos:
        print(f'Ano {ano}: todos já processados!')
        return

    log_buf = []
    ok = err = 0
    t0 = time.time()

    with tqdm(total=len(todos), desc=f'Chunking {ano}', unit='doc', dynamic_ncols=True) as pb:
        for md_path in todos:
            # --- PROTEÇÃO: ARQUIVOS VAZIOS OU CORROMPIDOS ---
            if not md_path.exists() or md_path.stat().st_size < 100:
                log_buf.append({
                    "doc_id": md_path.stem,
                    "status": "vazio_ou_corrompido",
                    "coerencia_media": 0,
                    "tipo_doc": "N/A",
                    "n_l1": 0, "n_l2": 0,
                    "estrategia_l1": "N/A"
                })
                err += 1
                pb.update(1)
                continue

            meta_path = md_path.with_suffix('.meta.json')

            # O processar_documento internamente chamará a gate_chunk(chunk, tipo_doc)
            log, saida = processar_documento(md_path, meta_path)

            if saida:
                out = dir_chunks / f'{md_path.stem}.chunks.json'
                out.write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding='utf-8')
                ok += 1
            else:
                err += 1

            log_buf.append(log)
            pb.set_postfix({'ok': ok, 'err': err})
            pb.update(1)

            # Checkpoint para segurança contra quedas de sessão
            if len(log_buf) >= CHECKPOINT_EVERY:
                pd.DataFrame(log_buf).to_csv(log_path, mode='a', header=not log_path.exists(), index=False)
                log_buf.clear()

    # Salva o restante do buffer no final
    if log_buf:
        pd.DataFrame(log_buf).to_csv(log_path, mode='a', header=not log_path.exists(), index=False)

    h = (time.time() - t0) / 3600
    print(f'\nAno {ano} concluído em {h:.1f}h  |  OK:{ok:,}  Err:{err:,}')

# Execução
run_chunking(ANO)

In [ ]:
# Célula 9 — Relatório final

log_path = BASE / 'logs' / f'chunking_{ANO}.csv'
if not log_path.exists():
    print(f'Ano {ANO}: log não encontrado — execute a Célula 8 primeiro')
else:
    df = pd.read_csv(log_path)
    print(f'\nRELATÓRIO CHUNKING {ANO} | Total: {len(df):,}')
    for s, n in df['status'].value_counts().items():
        print(f'  {s:<15}: {n:>6,} ({100*n/len(df):.1f}%)')

    ok_df = df[df['status'] == 'ok']
    if len(ok_df):
        print(f'  Estratégias L1 : {ok_df["estrategia_l1"].value_counts().to_dict()}')
        print(f'  Tipos de doc   : {ok_df["tipo_doc"].value_counts().to_dict()}')
        print(f'  L1 médio       : {ok_df["n_l1"].mean():.1f} chunks/doc')
        print(f'  L2 médio       : {ok_df["n_l2"].mean():.1f} chunks/doc')
        print(f'  Coerência média (geral)  : {ok_df["coerencia_media"].mean():.3f}')
        # Coerência por tipo — métrica mais honesta
        for tipo, grp in ok_df.groupby('tipo_doc'):
            print(f'  Coerência média ({tipo:<10}): {grp["coerencia_media"].mean():.3f}  ({len(grp):,} docs)')
        print(f'  Tokens totais  : {ok_df["tokens_total"].sum():,}')
        total_chunks = ok_df['n_l0'].sum() + ok_df['n_l1'].sum() + ok_df['n_l2'].sum()
        print(f'  Total chunks   : {total_chunks:,}')

    err_df = df[df['status'] != 'ok']
    if len(err_df):
        err_df.to_csv(BASE / 'logs' / f'erros_chunking_{ANO}.csv', index=False)
        print(f'  Erros: {len(err_df)} — salvos em erros_chunking_{ANO}.csv')

print('\nPróxima etapa: Etapa 4 — Embedding e indexação vetorial')